# Routing workflow with Pydantic AI

Routing classifies the input and sends it to the right agent for specialized handling. The classification can be done by an LLM or a traditional model.

```mermaid
flowchart LR 
    In([In]) --> Router["LLM Call Router"]

    Router -->|Route 1| LLM1["LLM Call 1"]
    Router -->|Route 2| LLM2["LLM Call 2"]
    Router -->|Route 3| LLM3["LLM Call 3"]

    LLM1 --> Out([Out])
    LLM2 --> Out
    LLM3 --> Out
```

**Examples:**
- Classify complexity of question and adjust model depending on it
- Classify type of query and use specialized tools (e.g., indexes, prompts)

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Setup

In [ ]:
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel
from pydantic_ai import Agent

load_dotenv()

## Vanilla workflow

We use a router agent with structured output to classify the message, then dispatch to the appropriate specialized agent.

In [ ]:
class RouterOutput(BaseModel):
    category: Literal["write_article", "generate_table_of_contents", "review_article"]


router_agent = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "You are a helpful assistant. You will classify the message into one of the following categories: "
        "'write_article', 'generate_table_of_contents', 'review_article'."
    ),
    output_type=RouterOutput,
)

agent_writer = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a writer. You will write an article about the topic provided.",
)

agent_toc = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "You are an expert writer specialized in SEO. Provided with a topic, "
        "you will generate the table of contents for a short article."
    ),
)

agent_reviewer = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a writer. You will review the article for the topic provided.",
)


def run_workflow(message: str) -> str:
    router_output = router_agent.run_sync(f"Classify the message: {message}")
    category = router_output.output.category
    if category == "write_article":
        return agent_writer.run_sync(f"Write an article about {message}").output
    elif category == "generate_table_of_contents":
        return agent_toc.run_sync(
            f"Generate the table of contents of an article about {message}"
        ).output
    else:
        return agent_reviewer.run_sync(
            f"Review the article for the topic {message}"
        ).output

In [ ]:
toc = run_workflow("Generate a table of contents for an article about AI")
print(toc)

In [ ]:
review = run_workflow(
    "Review this post: 'There are times where there's no time, so you don't have time to write an article about it.'"
)
print(review)

## Exercise

Implement a routing workflow that takes the content of a PDF file and depending on the type of document, processes it in a different way. Mock the processing for now.